# 03 - Metrics & EDA

Derives match outcomes (win/loss) from the cleaned Rocket League data, then compares winners and losers across all teams using basic and tactical metrics.

In [46]:
import pandas as pd

# Load the cleaned Rocket League data from notebook 02
csv_path = "../data/processed/rocket_league/ewc2026_falcons_cleaned.csv"
matches = pd.read_csv(csv_path)

print(f"Loaded {len(matches)} rows")


Loaded 138 rows


In [47]:
## Sum goals per team, per match



matches.groupby(["replay_id", "team_name"])["goals"].sum()

replay_id                             team_name
075ad706-291c-4e83-861f-7ee191555dac  FLCN         1
                                      R8           2
0ad20079-435b-4a21-889a-e5e7ac42d327  FLCN         2
                                      M8           1
31848ff0-b4d6-448e-9bf8-c402182b06e0  FLCN         3
                                      M8           2
429db940-97b3-438b-a438-91fd7cf5a907  FLCN         4
                                      TM           5
5b23babf-e452-4fa1-b3a1-93c910176894  FLCN         3
                                      SSG          0
623b116f-6d2e-4329-a3eb-a58d45347cab  FLCN         2
                                      M8           0
64749355-0867-456d-9656-2501085a0903  FLCN         1
                                      M8           2
703177c8-1376-4508-8363-8efafe72ca55  FLCN         2
                                      TM           0
710d4e87-6dbd-4509-8a3a-c1002aae0dae  FLCN         1
                                      M8           

In [48]:
## Reshape into one row per match



matches_wide = matches.groupby(["replay_id", "team_name"])["goals"].sum().unstack()


In [49]:
## Find the winning team for each match



matches_wide["winner"] = matches_wide.idxmax(axis=1)
matches_wide

team_name,FLCN,KC,M8,R8,SSG,TM,winner
replay_id,,,,,,,
075ad706-291c-4e83-861f-7ee191555dac,1.0,NaN,NaN,2.0,NaN,NaN,R8
0ad20079-435b-4a21-889a-e5e7ac42d327,2.0,NaN,1.0,NaN,NaN,NaN,FLCN
31848ff0-b4d6-448e-9bf8-c402182b06e0,3.0,NaN,2.0,NaN,NaN,NaN,FLCN
429db940-97b3-438b-a438-91fd7cf5a907,4.0,NaN,NaN,NaN,NaN,5.0,TM
5b23babf-e452-4fa1-b3a1-93c910176894,3.0,NaN,NaN,NaN,0.0,NaN,FLCN
623b116f-6d2e-4329-a3eb-a58d45347cab,2.0,NaN,0.0,NaN,NaN,NaN,FLCN
64749355-0867-456d-9656-2501085a0903,1.0,NaN,2.0,NaN,NaN,NaN,M8
703177c8-1376-4508-8363-8efafe72ca55,2.0,NaN,NaN,NaN,NaN,0.0,FLCN
710d4e87-6dbd-4509-8a3a-c1002aae0dae,1.0,NaN,3.0,NaN,NaN,NaN,M8


In [50]:
## Bring the winner back into the long table



matches = matches.merge(matches_wide[["winner"]], on="replay_id")
matches.head()

,source_group,replay_id,match_date,team_name,team_color,player_name,goals,assists,saves,shots,...,avg_speed,boost_stolen,pct_zero_boost,pct_supersonic,pct_defensive_third,pct_offensive_third,dist_to_mates,demos_inflicted,demos_taken,winner
0,flcn-vs-m8-qszrcck260,31848ff0-b4d6-448e-9bf8-c402182b06e0,2026-08-13 13:47:29+02:00,FLCN,blue,dralii,2,0,1,5,...,1530,764,11.047958,17.110874,46.354210,26.798140,3837,1,2,FLCN
1,flcn-vs-m8-qszrcck260,31848ff0-b4d6-448e-9bf8-c402182b06e0,2026-08-13 13:47:29+02:00,FLCN,blue,Rw9,1,1,2,4,...,1699,434,10.046940,24.956396,46.298060,18.309399,3734,3,1,FLCN
2,flcn-vs-m8-qszrcck260,31848ff0-b4d6-448e-9bf8-c402182b06e0,2026-08-13 13:47:29+02:00,FLCN,blue,Kiileerrz,0,1,0,5,...,1524,586,6.732835,12.711911,43.201145,22.106740,3657,0,0,FLCN
3,flcn-vs-m8-qszrcck260,31848ff0-b4d6-448e-9bf8-c402182b06e0,2026-08-13 13:47:29+02:00,M8,orange,Oski,1,1,3,1,...,1457,288,15.928627,9.358281,58.938705,16.353909,3738,1,1,FLCN
4,flcn-vs-m8-qszrcck260,31848ff0-b4d6-448e-9bf8-c402182b06e0,2026-08-13 13:47:29+02:00,M8,orange,nass,0,0,4,2,...,1457,232,14.972853,12.864300,53.259170,12.306739,3764,0,0,FLCN


In [51]:
## Flag whether each team won its match



matches["won"] = matches["team_name"] == matches["winner"]
matches.head()

,source_group,replay_id,match_date,team_name,team_color,player_name,goals,assists,saves,shots,...,boost_stolen,pct_zero_boost,pct_supersonic,pct_defensive_third,pct_offensive_third,dist_to_mates,demos_inflicted,demos_taken,winner,won
0,flcn-vs-m8-qszrcck260,31848ff0-b4d6-448e-9bf8-c402182b06e0,2026-08-13 13:47:29+02:00,FLCN,blue,dralii,2,0,1,5,...,764,11.047958,17.110874,46.354210,26.798140,3837,1,2,FLCN,True
1,flcn-vs-m8-qszrcck260,31848ff0-b4d6-448e-9bf8-c402182b06e0,2026-08-13 13:47:29+02:00,FLCN,blue,Rw9,1,1,2,4,...,434,10.046940,24.956396,46.298060,18.309399,3734,3,1,FLCN,True
2,flcn-vs-m8-qszrcck260,31848ff0-b4d6-448e-9bf8-c402182b06e0,2026-08-13 13:47:29+02:00,FLCN,blue,Kiileerrz,0,1,0,5,...,586,6.732835,12.711911,43.201145,22.106740,3657,0,0,FLCN,True
3,flcn-vs-m8-qszrcck260,31848ff0-b4d6-448e-9bf8-c402182b06e0,2026-08-13 13:47:29+02:00,M8,orange,Oski,1,1,3,1,...,288,15.928627,9.358281,58.938705,16.353909,3738,1,1,FLCN,False
4,flcn-vs-m8-qszrcck260,31848ff0-b4d6-448e-9bf8-c402182b06e0,2026-08-13 13:47:29+02:00,M8,orange,nass,0,0,4,2,...,232,14.972853,12.864300,53.259170,12.306739,3764,0,0,FLCN,False


In [52]:
## Confirm the new columns arrived and have no missing values



new_cols = ["boost_stolen", "pct_zero_boost", "pct_supersonic", "pct_defensive_third",
            "pct_offensive_third", "dist_to_mates", "demos_inflicted", "demos_taken"]
print(matches.shape)
matches[new_cols].isnull().sum()

(138, 22)


boost_stolen           0
pct_zero_boost         0
pct_supersonic         0
pct_defensive_third    0
pct_offensive_third    0
dist_to_mates          0
demos_inflicted        0
demos_taken            0
dtype: int64

In [53]:
## Define the metrics to compare (goals excluded: "won" is derived from it)



metrics = ["boost_bpm", "avg_speed", "saves", "shots"] + new_cols

In [54]:
## One row per team per match: average the 3 players of each team



team_match = matches.groupby(["replay_id", "team_name", "won"])[metrics].mean().reset_index()
print(team_match.shape)
print(team_match["won"].value_counts())

(46, 15)
won
False    23
True     23
Name: count, dtype: int64


In [55]:
## Winners vs losers across all teams



team_match.groupby("won")[metrics].agg(["mean", "std"])

boost_bpm               avg_speed                saves            \
             mean        std         mean        std      mean       std   
won                                                                        
False  448.376812  23.165697  1588.333333  51.549819  1.681159  0.843743   
True   440.652174  32.168508  1575.492754  62.518835  1.855072  0.716516   

          shots           boost_stolen             ... pct_defensive_third  \
           mean       std         mean        std  ...                mean   
won                                                ...                       
False  2.536232  0.914193   604.057971  179.24952  ...           46.408862   
True   3.028986  1.163038   528.173913  142.09802  ...           49.978499   

                pct_offensive_third           dist_to_mates              \
            std                mean       std          mean         std   
won                                                                       
False  7.699000           20.749663  5.647676   3631.405797  141.270603   
True   6.643341           19.214649  3.799336   3693.985507  179.490651   

      demos_inflicted           demos_taken            
                 mean       std        mean       std  
won                                                    
False         1.26087  0.619192     1.26087  0.724440  
True          1.26087  0.724440     1.26087  0.619192  

[2 rows x 24 columns]

In [56]:
## Standardized difference per metric (positive = winners higher)



means = team_match.groupby("won")[metrics].mean()
stds = team_match.groupby("won")[metrics].std()
pooled_std = ((stds.loc[True] ** 2 + stds.loc[False] ** 2) / 2) ** 0.5
effect = (means.loc[True] - means.loc[False]) / pooled_std
effect.round(2).sort_values(key=abs, ascending=False)

pct_defensive_third    0.50
shots                  0.47
boost_stolen          -0.47
dist_to_mates          0.39
pct_zero_boost        -0.34
pct_offensive_third   -0.32
boost_bpm             -0.28
avg_speed             -0.22
saves                  0.22
pct_supersonic        -0.16
demos_inflicted        0.00
demos_taken            0.00
dtype: float64

In [57]:
## Save the tables the dashboard will be built from



matches.to_csv("../data/processed/rocket_league/ewc2026_falcons_with_results.csv", index=False)
team_match.to_csv("../outputs/data_spec_exports/team_match.csv", index=False)
effect.round(2).sort_values(key=abs, ascending=False).rename("effect_size").to_csv(
    "../outputs/data_spec_exports/effect_sizes.csv", index_label="metric")

In [58]:
## Team summary: average of each metric per team



team_summary = team_match.groupby("team_name")[metrics].mean()
team_summary["n_matches"] = team_match.groupby("team_name").size()
team_summary = team_summary.round(2).reset_index()
team_summary.to_csv("../outputs/data_spec_exports/team_summary.csv", index=False)
print(team_summary.shape)
team_summary




(6, 14)


,team_name,boost_bpm,avg_speed,saves,shots,boost_stolen,pct_zero_boost,pct_supersonic,pct_defensive_third,pct_offensive_third,dist_to_mates,demos_inflicted,demos_taken,n_matches
0,FLCN,437.12,1589.43,1.84,2.90,538.45,11.81,18.39,49.23,19.52,3725.06,1.36,1.16,23
1,KC,465.00,1628.33,1.42,3.25,790.67,10.92,19.56,38.36,27.40,3681.42,1.08,0.75,4
2,M8,414.13,1480.93,1.87,2.27,524.13,13.13,13.21,51.03,18.71,3727.67,0.53,1.47,5
3,R8,450.50,1553.08,2.42,2.33,558.33,11.17,18.19,51.85,16.34,3581.08,1.75,1.58,4
4,SSG,436.33,1574.50,1.33,2.67,600.92,11.16,15.96,46.08,20.51,3636.42,1.25,1.58,4
5,TM,486.00,1630.44,1.50,2.83,539.44,14.40,21.61,47.38,19.93,3428.94,1.28,1.39,6


In [59]:
## Games table: one row per game, with date, opponent, score and result



team_cols = [c for c in matches_wide.columns if c not in ("winner", "FLCN")]
games = pd.DataFrame({
    "match_date": matches.groupby("replay_id")["match_date"].first(),
    "series": matches.groupby("replay_id")["source_group"].first(),
    "opponent": matches_wide[team_cols].idxmax(axis=1),
    "falcons_goals": matches_wide["FLCN"].astype(int),
    "opponent_goals": matches_wide[team_cols].max(axis=1).astype(int),
    "falcons_won": matches_wide["winner"] == "FLCN",
}).reset_index()
games = games.sort_values("match_date").reset_index(drop=True)
games.to_csv("../outputs/data_spec_exports/games.csv", index=False)
print(games.shape)
print("Falcons wins:", games["falcons_won"].sum())
games

(23, 7)
Falcons wins: 18


,replay_id,match_date,series,opponent,falcons_goals,opponent_goals,falcons_won
0,923c494c-78f1-4f5a-9493-e8d3e7a4b92e,2026-08-12 14:14:54+02:00,r8-vs-flcn-4jeuhpkpdy,R8,1,0,True
1,860186a2-6f99-4e5b-aa1d-e84b33fd4a9d,2026-08-12 14:24:16+02:00,r8-vs-flcn-4jeuhpkpdy,R8,2,1,True
2,075ad706-291c-4e83-861f-7ee191555dac,2026-08-12 14:31:48+02:00,r8-vs-flcn-4jeuhpkpdy,R8,1,2,False
3,e7e3da69-0a47-4c43-b47a-e16527e8d46b,2026-08-12 14:40:12+02:00,r8-vs-flcn-4jeuhpkpdy,R8,5,1,True
4,623b116f-6d2e-4329-a3eb-a58d45347cab,2026-08-13 13:11:54+02:00,flcn-vs-m8-qszrcck260,M8,2,0,True
5,0ad20079-435b-4a21-889a-e5e7ac42d327,2026-08-13 13:19:53+02:00,flcn-vs-m8-qszrcck260,M8,2,1,True
6,710d4e87-6dbd-4509-8a3a-c1002aae0dae,2026-08-13 13:27:49+02:00,flcn-vs-m8-qszrcck260,M8,1,3,False
7,64749355-0867-456d-9656-2501085a0903,2026-08-13 13:37:52+02:00,flcn-vs-m8-qszrcck260,M8,1,2,False
8,31848ff0-b4d6-448e-9bf8-c402182b06e0,2026-08-13 13:47:29+02:00,flcn-vs-m8-qszrcck260,M8,3,2,True
9,703177c8-1376-4508-8363-8efafe72ca55,2026-08-15 18:51:09+02:00,flcn-vs-tm-37y0c4e0wi,TM,2,0,True


In [60]:
## Series table: one row per series (one per opponent), with the score



series = games.groupby(["series", "opponent"]).agg(
    first_game=("match_date", "min"),
    games_played=("falcons_won", "size"),
    falcons_wins=("falcons_won", "sum"),
).reset_index()
series["falcons_losses"] = series["games_played"] - series["falcons_wins"]
series["score"] = series["falcons_wins"].astype(str) + "-" + series["falcons_losses"].astype(str)
series = series.sort_values("first_game").reset_index(drop=True)
series.to_csv("../outputs/data_spec_exports/series.csv", index=False)
print(series.shape)
series

(5, 7)


,series,opponent,first_game,games_played,falcons_wins,falcons_losses,score
0,r8-vs-flcn-4jeuhpkpdy,R8,2026-08-12 14:14:54+02:00,4,3,1,3-1
1,flcn-vs-m8-qszrcck260,M8,2026-08-13 13:11:54+02:00,5,3,2,3-2
2,flcn-vs-tm-37y0c4e0wi,TM,2026-08-15 18:51:09+02:00,6,4,2,4-2
3,kc-vs-flcn-1fbqk0yar1,KC,2026-08-16 17:35:38+02:00,4,4,0,4-0
4,grand-finals-tams2b1l9y,SSG,2026-08-16 19:49:57+02:00,4,4,0,4-0


## Findings (Rocket League, EWC 2026, 23 matches)

- No single metric clearly separates winners from losers. The largest standardized difference is 0.50 (pct_defensive_third); about 0.6 would be needed to stand out from chance with 23 vs 23 matches.
- Weak, consistent hints (hypotheses, not conclusions): winners use boost more economically (pct_zero_boost -0.34, boost_bpm -0.28) and spend more time in their own half (pct_defensive_third +0.50, pct_offensive_third -0.32, boost_stolen -0.47). Winners are also slightly more spread out (dist_to_mates +0.39).
- The positioning metrics are related to each other, so they describe roughly one pattern, not independent findings.
- Speed, saves and supersonic time barely differ (0.22 or less).
- Shots are higher for winners (+0.47), but this is nearly circular: shots produce goals.
- Demos are symmetric (every demo has an attacker and a victim), so they add nothing here.
- Limits: 23 matches from one tournament, all of them involving Falcons. Each match gives one winner and one loser, so the two groups are not independent (a paired comparison would be stricter).